# Uncut meshes for all surfaces
This notebook generates triangulated mesh surfaces for all fault surfaces in the Kanto fault dataset, without applying any mesh cutting. Meshes are written as VTK and OBJ files, and depth contours are written as GeoJSON files.

## Imports

In [1]:
import os
import numpy as np
from fault_mesh.faults.leapfrog import LeapfrogMultiFault

## Read fault data
Set the output coordinate system EPSG code and read fault data from the GeoJSON file.

In [2]:
# Set output coordinate system EPSG code (set to None to disable reprojection)
epsg = 32654

# Read fault data from GeoJSON file
fault_fname = r"fault_Kanto\fault_Kanto\Kanto_251216_Emori_crdchanged_FaultstatAdded_densified_500.geojson"

# Build fault network: set segment connection tolerance and find connections between segments
dist_tolerance = 1000.
fault_data = LeapfrogMultiFault.from_nz_cfm_shp(fault_fname, remove_colons=True, epsg=epsg, smoothing_n=None, dip_multiplier=1.0, exclude_zero=False)
fault_data.segment_distance_tolerance = dist_tolerance
fault_data.find_connections(verbose=False)

missing expected field
missing expected field
missing expected field


Found 5 connections
Found 4 connections between segment ends


## Define fault systems and cutting hierarchy
Read the manually-edited fault system definitions and generate curated multi-segment faults, then read the cutting hierarchy.

In [3]:
# Read fault system definitions and generate curated (multi-segment) faults
fault_data.read_fault_systems(r"fault_Kanto\fault_Kanto\Kanto_251216_Emori_251226_connected_v1_suggested.csv")
fault_data.generate_curated_faults()

# Read fault cutting hierarchy to determine termination relationships
fault_data.read_cutting_hierarchy(r"fault_Kanto\fault_Kanto\Kanto_251216_Emori_251226_hierarchy_v1_suggested.csv")

Creating connected fault system: Fukaya - Takasaki combined with segments: ['Fukaya', 'Takasaki']
Setting Takasaki as first segment
Creating connected fault system: Susugatani - Isehara combined with segments: ['Isehara', 'Susugatani']
Setting Susugatani as first segment
Creating connected fault system: Tachikawa-B - Tachikawa-A combined with segments: ['Tachikawa-A', 'Tachikawa-B']
Setting Tachikawa-B as first segment


## Create output directories
Create directories to hold the output mesh files and contour GeoJSON files.

In [4]:
for dir_name in ["test_vtks", "test_contours", "test_objs"]:
    if not os.path.exists(dir_name):
        os.mkdir(dir_name)

## Generate and write meshes
For each curated fault, generate depth contours then mesh the fault surface. If meshing fails, fall back to simple contour meshing and save the failed contours for inspection. Depth contours, VTK meshes, and OBJ meshes are all written out.

In [5]:
for fault in fault_data.curated_faults:
    print(fault.name)
    try:
        fault.generate_depth_contours(np.arange(0., 32000., 500.), smoothing=False)
        mesh = fault.mesh_fault_surface(check_mesh=False, resolution=500.)
    except Exception as e:
        # On meshing failure, fall back to simple contour meshing and save failed contours for inspection
        print(f"Failed to mesh {fault.name}: {e}")
        fault.contours.to_file(f"failed_contours_{fault.name}.geojson", driver="GeoJSON")
        mesh = fault.mesh_simple_contours(np.arange(0., 32000., 500.))
    # Write depth contours as GeoJSON and mesh as VTK and OBJ for inspection
    vtk_file = os.path.join("test_vtks", f"{fault.name}_depth_contours.vtk")
    obj_file = os.path.join("test_objs", f"{fault.name}_depth_contours.obj")
    geojson_file = os.path.join("test_contours", f"{fault.name}_depth_contours.geojson")
    fault.contours.to_file(geojson_file, driver="GeoJSON")
    mesh.write(vtk_file)
    mesh.write(obj_file)

Fukaya - Takasaki combined
Susugatani - Isehara combined
Tachikawa-B - Tachikawa-A combined
Ayasegawa
Hirai
Kamogawachikotai-minami
Kannawa
Kitatake
Kouzu-Matsuda
Kurokura
Minamishitaura
Sekiya
Shibusawa
